# Titanic 생존 예측 학습 및 검증

위에서 아래로 실행하면 데이터와 학습 코드를 검사한 뒤 모델을 학습함. `test.csv`에는 정답이 없어 훈련과 검증에 사용하지 않음.

In [ ]:
import math
import sys
import unittest
from datetime import datetime
from pathlib import Path
from types import SimpleNamespace

import torch
from torch import nn, optim
from torch.utils.data import DataLoader
import wandb

project_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "_03_homeworks/homework_2/titanic_dataset.py").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("프로젝트 디렉터리 안에서 노트북을 실행해야 함")
sys.path.insert(0, str(project_root))

from _03_homeworks.homework_2.titanic_dataset import get_preprocessed_dataset

## 데이터 로딩과 모델 학습 코드

In [ ]:
def get_data(run):
  train_dataset, validation_dataset, _ = get_preprocessed_dataset()
  print(len(train_dataset), len(validation_dataset))

  train_data_loader = DataLoader(dataset=train_dataset, batch_size=run.config.batch_size, shuffle=True)
  validation_data_loader = DataLoader(dataset=validation_dataset, batch_size=len(validation_dataset))

  return train_data_loader, validation_data_loader


class MyModel(nn.Module):
  def __init__(self, n_input, n_output, run):
    super().__init__()

    self.model = nn.Sequential(
      nn.Linear(n_input, run.config.n_hidden_unit_list[0]),
      nn.ReLU(),
      nn.Linear(run.config.n_hidden_unit_list[0], run.config.n_hidden_unit_list[1]),
      nn.ReLU(),
      nn.Linear(run.config.n_hidden_unit_list[1], n_output),
    )

  def forward(self, x):
    x = self.model(x)
    return x


def get_model_and_optimizer(run):
  my_model = MyModel(n_input=10, n_output=1, run=run)
  optimizer = optim.SGD(my_model.parameters(), lr=run.config.learning_rate)

  return my_model, optimizer


def training_loop(model, optimizer, train_data_loader, validation_data_loader, run):
  n_epochs = run.config.epochs
  loss_fn = nn.BCEWithLogitsLoss()
  next_print_epoch = 100

  for epoch in range(1, n_epochs + 1):
    loss_train = 0.0
    num_trains = 0
    for train_batch in train_data_loader:
      input = train_batch['input']
      target = train_batch['target'].float().unsqueeze(1)
      output_train = model(input)
      loss = loss_fn(output_train, target)
      loss_train += loss.item()
      num_trains += 1

      optimizer.zero_grad()
      loss.backward()
      optimizer.step()

    loss_validation = 0.0
    num_validations = 0
    with torch.no_grad():
      for validation_batch in validation_data_loader:
        input = validation_batch['input']
        target = validation_batch['target'].float().unsqueeze(1)
        output_validation = model(input)
        loss = loss_fn(output_validation, target)
        loss_validation += loss.item()
        num_validations += 1

    run.log({
      "Epoch": epoch,
      "Training loss": loss_train / num_trains,
      "Validation loss": loss_validation / num_validations
    })

    if epoch >= next_print_epoch:
      print(
        f"Epoch {epoch}, "
        f"Training loss {loss_train / num_trains:.4f}, "
        f"Validation loss {loss_validation / num_validations:.4f}"
      )
      next_print_epoch += 100


def main(args):
  current_time_str = datetime.now().astimezone().strftime('%Y-%m-%d_%H-%M-%S')

  config = {
    'epochs': args.epochs,
    'batch_size': args.batch_size,
    'learning_rate': 1e-3,
    'n_hidden_unit_list': [20, 20],
  }

  with wandb.init(
    mode="online" if args.wandb else "disabled",
    project="my_model_training",
    notes="Titanic survival classification",
    tags=["my_model", "titanic"],
    name=current_time_str,
    config=config
  ) as run:
    print(args)
    print(run.config)

    # 가로축을 Epoch, 세로축을 Training loss로 그려줘.”
    run.define_metric("Training loss", step_metric="Epoch")

    # 가로축을 Epoch, 세로축을 Validation loss로 그려줘, 실험 요약에는 가장 낮았던 값을 남겨줘.”
    run.define_metric("Validation loss", step_metric="Epoch", summary="min")

    train_data_loader, validation_data_loader = get_data(run)

    linear_model, optimizer = get_model_and_optimizer(run)

    print("#" * 50, 1)

    training_loop(
      model=linear_model,
      optimizer=optimizer,
      train_data_loader=train_data_loader,
      validation_data_loader=validation_data_loader,
      run=run
    )

## 데이터 형식과 한 epoch 학습 검사

In [ ]:
class FakeRun:
  def __init__(self):
    self.config = SimpleNamespace(
      batch_size=2,
      epochs=1,
      learning_rate=1e-3,
      n_hidden_unit_list=[8, 8],
    )
    self.logged = []

  def log(self, metrics):
    self.logged.append(metrics)


class TitanicTrainingTest(unittest.TestCase):
  def test_data_loaders_use_titanic_classification_samples(self):
    train_loader, validation_loader = get_data(FakeRun())

    self.assertEqual(len(train_loader.dataset) + len(validation_loader.dataset), 891)
    batch = next(iter(train_loader))
    self.assertEqual(batch["input"].shape[1], 10)
    self.assertEqual(batch["target"].dtype, torch.int64)

  def test_one_epoch_logs_finite_classification_loss(self):
    run = FakeRun()
    samples = [
      {"input": torch.zeros(10), "target": torch.tensor(0)},
      {"input": torch.ones(10), "target": torch.tensor(1)},
      {"input": torch.full((10,), 2.0), "target": torch.tensor(0)},
    ]
    loader = DataLoader(samples, batch_size=2)
    model, optimizer = get_model_and_optimizer(run)

    self.assertEqual(model(torch.zeros(2, 10)).shape, (2, 1))
    self.assertIsInstance(optimizer, torch.optim.SGD)
    training_loop(model, optimizer, loader, loader, run)

    self.assertEqual(len(run.logged), 1)
    metrics = run.logged[0]
    self.assertEqual(metrics["Epoch"], 1)
    self.assertEqual(set(metrics), {"Epoch", "Training loss", "Validation loss"})
    self.assertTrue(math.isfinite(metrics["Training loss"]))
    self.assertTrue(math.isfinite(metrics["Validation loss"]))

suite = unittest.defaultTestLoader.loadTestsFromTestCase(TitanicTrainingTest)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), "학습 코드 검사 실패"

## 학습 설정

`USE_WANDB = True`로 바꾸면 W&B에 지표를 기록함.

In [ ]:
EPOCHS = 1_000
BATCH_SIZE = 512
USE_WANDB = False

## 모델 학습

In [ ]:
args = SimpleNamespace(epochs=EPOCHS, batch_size=BATCH_SIZE, wandb=USE_WANDB)
main(args)